# 03 · Classwork: build an "Ask" tool

Goal: a small tool you can run from the terminal:

```
python ask.py "What is a token?"
```

Build and test each piece here in the notebook first, then copy the finished functions into `ask.py`.

**Requirements**

1. Load the key, base_url and model from `.env`
2. Use a system prompt you write yourself
3. Print the answer, tokens in and out, and latency
4. Retry on 429, timeouts and connection errors
5. Log every call to `logs/calls.jsonl`
6. Push to GitHub with a README and `.gitignore`

**Stretch:** a `--provider groq` option that switches provider.

In [1]:
import os
import time
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()  # reads the .env file in this folder

GEMINI_BASE_URL = "https://generativelanguage.googleapis.com/v1beta/openai/"
GROQ_BASE_URL = "https://api.groq.com/openai/v1"

GEMINI_MODEL = os.getenv("GEMINI_MODEL", "gemini-3.8-flash")
GROQ_MODEL = os.getenv("GROQ_MODEL", "openai/gpt-oss-20b")

# Never print the key itself, only whether it was found
print("Gemini key found:", bool(os.getenv("GEMINI_API_KEY")))
print("Groq key found:  ", bool(os.getenv("GROQ_API_KEY")))

Gemini key found: True
Groq key found:   True


## Task 1 · Write your system prompt

In [2]:
# Task 1 — system prompt with a clear role and rules
SYSTEM_PROMPT = (
    "You are a concise study assistant for computer science students. "
    "Answer in plain English in at most 4 sentences. "
    "If you are not sure of an answer, say 'I don't know' instead of guessing. "
    "Never ask the user to repeat the question."
)

print(SYSTEM_PROMPT)

You are a concise study assistant for computer science students. Answer in plain English in at most 4 sentences. If you are not sure of an answer, say 'I don't know' instead of guessing. Never ask the user to repeat the question.


## Task 2 · Create the client and an `ask()` function with retries

Tip: notebook 02 has a working example.

In [4]:
from openai import RateLimitError, APITimeoutError, APIConnectionError

RETRYABLE = (RateLimitError, APITimeoutError, APIConnectionError)

# max_retries=0 so we handle retries ourselves
client = OpenAI(api_key=os.getenv("GROQ_API_KEY"), base_url=GROQ_BASE_URL, max_retries=0)


def ask(question, retries=3, timeout=30):
    """Send SYSTEM_PROMPT + question; retry transient errors with back-off (2s, 4s, 8s).
    Returns the full response object."""
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": question},
    ]
    for attempt in range(1, retries + 1):
        try:
            return client.chat.completions.create(
                model=GROQ_MODEL, messages=messages, timeout=timeout
            )
        except RETRYABLE as e:
            wait = 2 ** attempt
            print(f"Retry {attempt} in {wait}s: {type(e).__name__}")
            time.sleep(wait)
    raise RuntimeError("All retries failed")


# quick test
r = ask("What is a token in LLMs?")
print(r.choices[0].message.content)

In the context of large language models (LLMs), a token is a unit of text that the model processes, which can be a word, part of a word, or even a punctuation mark. The model’s tokenizer splits a sentence into these tokens before feeding them into the network. Tokens are the basic building blocks that the model learns to predict, so each token’s representation influences the next token’s output. The size of the token vocabulary determines how many distinct tokens the model can handle.


## Task 3 · Print the answer, tokens and latency

In [5]:
start = time.perf_counter()
response = ask("What is the difference between a tuple and a list?")
latency = time.perf_counter() - start

print(response.choices[0].message.content)
print("-" * 40)
print("tokens in: ", response.usage.prompt_tokens)
print("tokens out:", response.usage.completion_tokens)
print(f"latency:    {latency:.2f}s")

In many languages, a list is mutable while a tuple is immutable. Lists are defined with brackets `[]` and tuples with parentheses `()`. Because tuples can’t be changed, they can be used as dictionary keys and may be slightly faster for read‑only data. In contrast, lists support operations such as `append`, `remove`, and `sort`.
----------------------------------------
tokens in:  135
tokens out: 338
latency:    0.81s


## Task 4 · Log every call to logs/calls.jsonl

In [6]:
import json
import datetime
from pathlib import Path

LOG_FILE = Path("logs/calls.jsonl")
LOG_FILE.parent.mkdir(exist_ok=True)


def log_call(model, usage, latency, ok=True, error=None, provider="groq"):
    """Append one JSON line per call. Never log API keys or personal data."""
    record = {
        "time": datetime.datetime.now().isoformat(timespec="seconds"),
        "provider": provider,
        "model": model,
        "tokens_in": usage.prompt_tokens if usage else None,
        "tokens_out": usage.completion_tokens if usage else None,
        "latency_s": round(latency, 2),
        "ok": ok,
        "error": error,
    }
    with open(LOG_FILE, "a") as f:
        f.write(json.dumps(record) + "\n")


def ask_and_log(question, provider="groq"):
    """ask() + logging, on success and on failure. Returns the response object."""
    start = time.perf_counter()
    try:
        r = ask(question)
        log_call(GROQ_MODEL, r.usage, time.perf_counter() - start, provider=provider)
        return r
    except Exception as e:
        log_call(GROQ_MODEL, None, time.perf_counter() - start, ok=False,
                 error=type(e).__name__, provider=provider)
        raise


# test: every call below leaves one line in logs/calls.jsonl
r = ask_and_log("What does HTTP status 429 mean? One sentence.")
print(r.choices[0].message.content)
print("\nLogged to", LOG_FILE)

HTTP status 429 means “Too Many Requests” – the server is telling the client it has exceeded the allowed request rate.

Logged to logs\calls.jsonl


## Task 5 · Turn it into ask.py

Create `ask.py` next to this notebook. Read the question from the command line:

```python
import sys
question = sys.argv[1]
```

Run it in the VS Code terminal (with your `.venv` active):

```
python ask.py "What is a token?"
```

Then commit and push. Check that `.env` is **not** in `git status` before you commit.

## All tasks done

The finished, tested version of these functions is in `ask.py` in this folder —
including the stretch goal: `--provider gemini`. See Task 5 for how to run it.